# Which features are the most predictive in each modality/model?

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
from statannot import add_stat_annotation
import joblib
from sklearn import inspection

from importlib import reload
import sys
import os
sys.path.insert(1,'../')
import plots

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/noOsteo'

In [ ]:
def plot_features_CI(param,params,ax,ci=95):
    zscore = stats.norm.ppf(1-(1-ci/100)/2)
    print(zscore,ci)
    params = params[params['mean']!=0]
    params = params.sort_values('mean')
    params['CI'] = params['std']/np.sqrt(5)*zscore
    params = params[(np.abs(params['mean'])-params['CI'])>0].reset_index()
    param['sign'] = 'not significant'
    param = param.set_index('index')
    param.loc[params.loc[params['mean']<0,'index'],'sign'] = 'protective'
    param.loc[params.loc[params['mean']>0,'index'],'sign'] = 'harmful'
    param = param.loc[params['index']].reset_index()
    param['f_cat'] = pd.Categorical(
        param['index'], 
        categories=params['index'].values, 
        ordered=True)
    param = param.sort_values('f_cat')
    param['index'] = param['index'].replace(dict_acc)
    param['index'] = param['index'].replace(dict_covs)
    print(param)
    if param.shape[0] > 0:
        ax1 = sns.pointplot(x=0,y='index',data=param,ci=ci,join=False,hue='sign',palette=['blue','red','grey'],
                          hue_order=['protective','harmful'],ax=ax)
        ax1.axvline(0,0,13,color='k',linestyle='--')
        ax1.set_xlabel('regression coefficient')
        ax1.set_ylabel('')

In [ ]:
dict_acc = {'mean_N_sedentary_intervals_23-07':'how often "sedentary" during night','mean_N_light_intervals_23-07':'how often "light" during night',
            'mean_N_light_intervals_07-23':'how often "light" during day',
            'mean_N_sleep_intervals_07-23':'how often "sleep" during day','mean_movement_during_light':'mean acceleration during "light"',
           'Wednesday_average_acceleration':'average acceleration on Wednesday','Average0200___0259':'average acceleration from 02:00 - 02:59',
            'mean_N_MVPA_intervals_per24h':'how often "MVPA" during 24h','max_sedentary_hours_consecutive':'maximum hours "sedentary" without disruption',
            'mean_N_imputed_intervals_per24h':'how often "imputed" during 24h','std_sleep_hours_per24h':'std of hours "sleep" per 24h',
           'No_wear_time_bias_adjustedmedian':'median acceleration adjusted','Average2300___2359':'average acceleration from 23:00 - 23:59',
           'Average1900___1959':'average acceleration from 19:00 - 19:59','std_sleep_hours_perday':'std of hours "sleep" per day',
           'max_sleep_hours_consecutive':'maxiumum time spent continuously asleep',
           'Average1700___1759':'average acceleration from 17:00 - 17:59',
           'mean_N_imputed_intervals_07-23':'how often "imputed" during day',
           'std_movement_during_light':'std of acceleration during "light"',
           'mean_max_sleep_hours_consecutive_per24h':'mean maxiumum time spent continuously asleep during 24h',
           'mean_N_sleep_intervals_per24h':'how often "sleep" during 24h',
           'mean_sedentary_hours_per24h':'how long "sedentary" during 24h',}
dict_covs = {'accelerometry_age':'age'}

In [ ]:
# feature importance consensus
diags = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']
diagnames = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDallHC','diag_ProdallHC','diag_PDProdallHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']
kinds = ['HC','HC','HC','allHC','allHC','allHC','pop','pop','pop']
features = ['genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD',
            'all_acc_features','all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
PRSs = pd.Index(['Age at menopause', "Alzheimer's disease",
       'Atrial fibrillation', 'Age-related macular degeneration', 'Asthma',
       'Breast cancer', 'Bipolar disorder', 'Body mass index',"Crohn's disease", 'Coeliac disease', 'Bowel cancer',
       'Cardiovascular disease UK', 'Estimated bone mineral density T-score',
       'Epithelial ovarian cancer', 'Glycated haemoglobin (diabetes free)',
       'HDL cholesterol', 'Height''Hypertension', 'Intraocular pressure', 'Ischaemic stroke',
       'LDL cholesterol (statin free)', 'Melanoma', 'Multiple sclerosis',
       'Osteoporosis', 'Prostate cancer', "Parkinson's disease", 'Psoriasis',
        'Rheumatoid arthritis', 'Schizophrenia', 'Systemic lupus erythematosus',
        'Type 1 diabetes', 'Type 2 diabetes', 'Ulcerative colitis','Venous thromboembolic disease'])
savefig2 = []

for j,kind,diag,diagname in zip(np.arange(len(diags)),kinds,diags,diagnames):
        savefig=f'{image_path}/selected_features_sign5fold_{diagname}'
        labels = ['A','B','C','D','E','F']
        figsingle,axessingle = plt.subplots(nrows=3,ncols=2,figsize=(20,20))
        plots.plot_context()
        for (i,f),ax,label in zip(enumerate(features),axessingle.flatten(),labels):
                if kind == 'pop':
                    path = f'{data_path}/{f}/{kind}modelsNoPD_params_stacked_allHC.csv'
                elif kind == 'allHC':
                    path = f'{data_path}/{f}/HCmodels_params_stacked_allHC.csv'
                elif kind == 'HC':
                    path = f'{data_path}/{f}/{kind}models_params_stacked_matched.csv'
                print(diag)

                param = pd.read_csv(path)
                if f=='genetics+family' or f==features[-1]:
                    param.rename(columns = dict(zip(PRSs, 'PRS_'+PRSs)), inplace=True)
                predictors = param.columns[2:]
                params = pd.DataFrame(index=predictors,columns=['mean','std'])
                param = param.dropna(axis='rows',how='all',subset=predictors)
                param = param[param['model']==diag]
                params.loc[predictors,'mean'] = param[predictors].mean(axis=0).values
                params.loc[predictors,'std'] = param[predictors].std(axis=0).values

                param = param.set_index(['model','cv_fold']).stack().droplevel(level=[0,1]).reset_index()
                params = params.sort_values('mean',ascending=True)
                plot_features_CI(param,params,ci=100-(0.05/len(predictors)),ax=ax)
                ax.text(-0.22, 1,label, transform=ax.transAxes,size=20, weight='bold')
                if i == 1:
                    ax.legend(loc=(1,1))
                else:
                    ax.legend([],[],frameon=False)
                if f == 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD' and kind=='pop' and False:
                    if j==6:
                        savefig2=f'{image_path}/selected_features_sign5fold_combined'
                        figc,axesc = plt.subplots(ncols=2,figsize=(7,3))
                        plots.plot_context()
                    if diagname == 'diag_PDPopulationNoProd':
                        plot_features_CI(param,params,ci=100-(0.05/len(predictors)),ax=axesc[0])
                        axesc[0].text(-0.22, 1,'A', transform=axesc[0].transAxes,size=20, weight='bold')
                        axesc[0].legend([],[],frameon=False)
                    elif diagname == 'diag_ProdPopulationNoPD':
                        plot_features_CI(param,params,ci=100-(0.05/len(predictors)),ax=axesc[1])
                        axesc[1].text(-0.22, 1,'B', transform=axesc[1].transAxes,size=20, weight='bold')
                        axesc[1].legend(loc=(1,1))


        if savefig:
            figsingle.tight_layout()
            figsingle.savefig(f'{savefig}.pdf',bbox_inches='tight',dpi=300)
            figsingle.savefig(f'{savefig}.png',bbox_inches='tight',dpi=300)
        if savefig2:
            figc.tight_layout()
            figc.savefig(f'{savefig2}.pdf',bbox_inches='tight',dpi=300)
            figc.savefig(f'{savefig2}.png',bbox_inches='tight',dpi=300)

In [ ]:
# stacked coefs
labels = ['A','B']
kinds = ['matched unaffected controls']
cols = sns.color_palette("tab10")[1:6]
cols.insert(0,(0,0,0))

cases = ['Prod','PD']
fig,axes = plt.subplots(ncols=2,figsize=(15,5))
plots.plot_context()
for (i,case),name,ax,label in zip(enumerate(cases),['prodromal','diagnosed'],axes.flatten(),labels):
    path = f'{data_path}/all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD/{case}HC_matched_stacked.csv'
    coefs = pd.read_csv(path,header=[0,1],index_col=0)
    temp = coefs.unstack().to_frame().reset_index()
    temp[0] = temp[0].astype(float)
    coef_pivot = pd.pivot_table(index = ['modality','level_2'], columns=['statistic'],values=0,data=temp).reset_index()
    coef_pivot['modality'] = coef_pivot['modality'].replace(['all_acc_features','prodromal_symptoms','genetics'],
                                                            ['all accelerometer','prodromal symptoms','genetics + family'])

    ax = sns.barplot(data=coef_pivot,y='modality',x='coef',color='k',ci=100-0.05/6,ax=ax,
                    order=['Intercept','genetics + family','lifestyle','blood','prodromal symptoms','all accelerometer'],
                     palette=cols)
    ax.set_xlabel('coefficient estimate')
    ax.set_ylabel('')
    ax.set_title(f'{name} vs\n{kinds[0]}')
    ax.text(-0.1, 1.05,label, transform=ax.transAxes,size=20, weight='bold')
plt.tight_layout()
plt.savefig(f'{image_path}/stackedmodel_coefs_matchedHC.pdf',dpi=300,bbox_inches='tight')
plt.savefig(f'{image_path}/stackedmodel_coefs_matchedHC.png',dpi=300,bbox_inches='tight')

In [ ]:
# stacked coefs
labels = ['A','B','C']
kinds = ['matched unaffected controls']#,'unaffected controls','population']
cols = sns.color_palette("tab10")[1:6]
cols.insert(0,(0,0,0))

cases = ['Prod','PD']
for (i,case),name in zip(enumerate(cases),['prodromal','diagnosed']):
    fig,axes = plt.figure(figsize=(15,5))
    plots.plot_context()
    for diag,ax,label,kind in zip([f'{case}HC',f'{case}allHC',f'{case}PopulationNo{cases[(i+1)%2]}'],np.arange(1),labels,kinds):
        if diag == 'ProdHC' or diag=='PDHC':
            path = f'{data_path}/all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD/{diag}_matched_stacked.csv'
        elif diag == 'ProdallHC' or diag=='PDallHC':
            path = f'{data_path}/all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD/{case}HC_allHC_stacked.csv'
        else:
            path = f'{data_path}/all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD/{diag}_allHC_stacked.csv'
        coefs = pd.read_csv(path,header=[0,1],index_col=0)
        temp = coefs.unstack().to_frame().reset_index()
        temp[0] = temp[0].astype(float)
        coef_pivot = pd.pivot_table(index = ['modality','level_2'], columns=['statistic'],values=0,data=temp).reset_index()
        coef_pivot['modality'] = coef_pivot['modality'].replace(['all_acc_features','prodromal_symptoms','genetics'],
                                                                ['all accelerometer','prodromal symptoms','genetics + family'])

        ax = sns.barplot(data=coef_pivot,y='modality',x='coef',color='k',ci=100-0.05/6,
                        order=['Intercept','genetics + family','lifestyle','blood','prodromal symptoms','all accelerometer'],
                         palette=cols)
        ax.set_xlabel('coefficient estimate')
        ax.set_ylabel('')
        ax.set_title(f'{name} vs\n{kind}')
        #ax.text(-0.1, 1.05,label, transform=ax.transAxes,size=20, weight='bold')
    plt.tight_layout()
    plt.savefig(f'{image_path}/{case}_stackedmodel_coefs.pdf',dpi=300,bbox_inches='tight')
    plt.savefig(f'{image_path}/{case}_stackedmodel_coefs.png',dpi=300,bbox_inches='tight')

In [ ]:
# stacked coefs
for diag in diags:
    print(diag)
    diag_name = diag.replace('diag_','')
    coefs = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/{diag_name}_stacked.csv',header=[0,1],index_col=0)
    temp = coefs.unstack().to_frame().reset_index()
    temp[0] = temp[0].astype(float)
    coef_pivot = pd.pivot_table(index = ['modality','level_2'], columns=['statistic'],values=0,data=temp).reset_index()
    coef_pivot['modality'] = coef_pivot['modality'].replace(['all_acc_features','prodromal_symptoms','genetics'],
                                                            ['all accelerometer','prodromal symptoms','genetics + family'])
    fig = plt.figure(figsize=(7,3))
    plots.plot_context()
    ax = sns.barplot(data=coef_pivot,y='modality',x='coef',color='k',ci=100-0.05/6)
    ax.set_xlabel('coefficient estimate')
    ax.set_ylabel('')
    plt.savefig(f'{image_path}/{diag}_stackedmodel_coefs.pdf',dpi=300,bbox_inches='tight')
    plt.savefig(f'{image_path}/{diag}_stackedmodel_coefs.png',dpi=300,bbox_inches='tight')